# 5. Hybrid Recommendation System ✅

Combines **Clustering (personalization)** + **Apriori Rules (patterns)** + **Content similarity**.

In [1]:
!pip install gradio scikit-learn mlxtend joblib --quiet

import pandas as pd
import numpy as np
import joblib
import gradio as gr
from sklearn.metrics.pairwise import cosine_similarity
import sys
sys.path.append('src')

print('✅ All packages ready!')

e:\Ecommerce Recommendation System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All packages ready!


In [2]:
# Load all models
print('Loading models...')

# Clustering
kmeans = joblib.load('models/kmeans_clusters.joblib')
scaler = joblib.load('models/feature_scaler.joblib')
clustered_users = pd.read_csv('data/processed/clustered_users.csv')

# Association rules
rules = joblib.load('models/apriori_rules.joblib')

# Product data
df_clean = pd.read_csv('data/processed/cleaned_user_products.csv')
products = df_clean['product_name'].unique()

print(f'✅ Loaded: {len(clustered_users)} users, {len(rules)} rules, {len(products)} products')

Loading models...
✅ Loaded: 206209 users, 353 rules, 1000 products


In [3]:
# Create product similarity (simple TF-IDF style)
from sklearn.feature_extraction.text import TfidfVectorizer

# Simple product embeddings using product name + department
df_clean['product_features'] = (df_clean['product_name'] + ' ' + 
                               df_clean['department_id'].astype(str))

vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
product_tfidf = vectorizer.fit_transform(df_clean['product_features'].unique())
product_sim = cosine_similarity(product_tfidf)
product_names = vectorizer.get_feature_names_out()

print(f'Product similarity matrix: {product_sim.shape}')

Product similarity matrix: (1000, 1000)


In [4]:
def hybrid_recommend(product_name, user_id=99999, top_k=5):
    """Hybrid recommendation: Rules + Cluster + Content"""
    recs = []
    scores = []
    product_name = product_name.lower().strip()
    
    print(f'🔍 Recommending for: {product_name}')
    
    # 1. ASSOCIATION RULES (60% weight)
    rule_recs = []
    for _, row in rules.iterrows():
        antecedents = ' '.join([str(i).lower() for i in row['antecedents']])
        if product_name in antecedents:
            rule_recs.extend(row['consequents'])
    
    # 2. CLUSTER PERSONALIZATION (20% weight)
    user_cluster = get_user_cluster(user_id)
    cluster_products = df_clean[df_clean['user_id'].isin(
        clustered_users[clustered_users['cluster'] == user_cluster]['user_id']
    )]['product_name'].value_counts().head(3).index.tolist()
    
    # 3. CONTENT SIMILARITY (20% weight)
    prod_idx = np.where(product_names == product_name)[0]
    if len(prod_idx) > 0:
        sim_scores = product_sim[prod_idx[0]]
        content_recs = [product_names[i] for i in np.argsort(sim_scores)[-4:] if product_names[i] != product_name]
    else:
        content_recs = []
    
    # Combine with weights
    all_recs = list(set(rule_recs + cluster_products + content_recs))
    
    # Score based on source
    for rec in all_recs:
        score = 0
        if rec in rule_recs:
            score += 0.6 * rules[rules['consequents'].apply(lambda x: rec in x)]['confidence'].mean()
        if rec in cluster_products:
            score += 0.2
        if rec in content_recs:
            score += 0.2 * sim_scores[list(product_names).index(rec)]
        if score > 0:
            recs.append(rec)
            scores.append(score)
    
    # Sort and return top k
    rec_df = pd.DataFrame({'product': recs, 'score': scores})
    top_recs = rec_df.nlargest(top_k, 'score')
    
    return top_recs['product'].tolist(), top_recs['score'].round(3).tolist()

def get_user_cluster(user_id):
    """Get user cluster or default to largest cluster"""
    user_row = clustered_users[clustered_users['user_id'] == user_id]
    if len(user_row) > 0:
        return user_row['cluster'].iloc[0]
    return clustered_users['cluster'].mode().iloc[0]

In [ ]:
# Test the hybrid recommender
print('🧪 Testing hybrid recommender...')
recs, scores = hybrid_recommend('milk')

print('\n🏆 Hybrid Recommendations:')
for i, (rec, score) in enumerate(zip(recs, scores), 1):
    print(f'{i}. {rec} (score: {score:.3f})')

🧪 Testing hybrid recommender...
🔍 Recommending for: milk

🏆 Hybrid Recommendations:
1. Banana (score: 0.347)
2. Bag of Organic Bananas (score: 0.336)
3. Organic Baby Spinach (score: 0.303)
4. heavenly (score: 0.196)
5. pretzel (score: 0.196)


🔍 Recommending for: garlic powder


: 

In [7]:
# Gradio Web UI
def gradio_interface(product_name, user_id):
    recs, scores = hybrid_recommend(product_name, int(user_id))
    result = '\n'.join([f'{i+1}. {r} (score: {s:.2f})' for i, (r, s) in enumerate(zip(recs, scores))])
    return result

demo = gr.Interface(
    fn=gradio_interface,
    inputs=[
        gr.Textbox(label='Product Name', placeholder='e.g., banana, organic strawberries'),
        gr.Textbox(label='User ID (optional)', placeholder='99999', value='99999')
    ],
    outputs=gr.Textbox(label='Recommendations', lines=8),
    title='🛒 Hybrid E-commerce Recommender',
    description='Powered by K-Means Clustering + Apriori Rules + Content Similarity',
    examples=[['banana', '99999'], ['strawberries', '12345'], ['milk', '67890']]
)

print('🚀 Launching Gradio interface...')
demo.launch(share=True)

🚀 Launching Gradio interface...
* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


🔍 Recommending for: milk
🔍 Recommending for: chocolate
🔍 Recommending for: distilled water
🔍 Recommending for: spaghetti pasta
🔍 Recommending for: organic baby spinach
🔍 Recommending for: strawberries
🔍 Recommending for: milk
🔍 Recommending for: strawberries
🔍 Recommending for: garlic powder


## 🎉 PROJECT COMPLETE! ✅

**Full Pipeline:**
1. **Data Preprocessing** → Processed datasets
2. **EDA** → Business insights
3. **Clustering** → 5 customer segments
4. **Apriori** → 100s of association rules
5. **Hybrid Recommender** → Personalized recommendations

**Production Ready:** Gradio UI live! 📱
**Models saved & deployable!** 🚀